<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/04_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/bilgeerol/bim448-sentiment/blob/main/notebooks/04_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 – LLM ile Duygu Analizi
**Sahibi:** Sude · **Amaç:** İngilizce ve Türkçe test setlerini `openai/gpt-oss-120b` (Groq, ücretsiz katman) ile sıfır örnekli olarak, 20'şerli gruplar hâlinde sınıflandırmak; sonuçları doğrulamak, hata örneklerini ve yöntemler arası uyumu çıkarmak.
**Girdi:** `data/processed/{en,tr}_val.csv`, `data/processed/{en,tr}_test.csv`, `predictions/{vader,bert}_*.csv`
**Çıktı:** `predictions/llm_{en,tr}.csv`, `results/logs/llm_*_raw.jsonl`, `results/logs/llm_pilot_raw.jsonl`, `results/logs/llm_calisma_ozeti.csv`, `results/tables/yontem_uyumu.csv`, `results/hata_ornekleri/llm.csv`
**Not:** Baştan sona çalıştırıldığında `_partial.csv` ve pilot log'u sayesinde API'ye yeni istek gitmez. Gerekli Colab Secrets: `GROQ_API_KEY`, `GITHUB_TOKEN`.

## 0. Kurulum

In [5]:
# 0. KURULUM HÜCRESİ — her oturumda ilk bu çalıştırılır
from google.colab import userdata, drive
import os
drive.mount('/content/drive')  # modeller ve ham veri Drive'da saklanır
TOKEN = userdata.get('GITHUB_TOKEN')
REPO_OWNER = "bilgeerol"
REPO = "bim448-sentiment"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone https://{TOKEN}@github.com/{REPO_OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git pull
!git config user.name "sudedereli"
!git config user.email "sudedereli04@gmail.com"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/bim448-sentiment
Already up to date.


## 1. Groq bağlantısı, istem metni ve ortak fonksiyonlar

In [6]:
# Groq istemcisi, sabitler, istem metni ve ortak fonksiyonlar (istek gönderme, doğrulama, ham kayıt, GitHub ara kaydı)
!pip install -q groq
import pandas as pd, json, time, os, re
import groq
from google.colab import userdata

client = groq.Groq(api_key=userdata.get("GROQ_API_KEY"))
MODEL = "openai/gpt-oss-120b"
REASONING = "low"
GRUP = 20
RPM, TPM = 30, 8000          # ücretsiz katman limitleri; plan değişirse bu iki sayı güncellenir
ETIKETLER = {"negative", "neutral", "positive"}

SYSTEM_PROMPT = """You are a sentiment classifier for product reviews. Each review is written
in English or Turkish. For each review, decide the reviewer's overall
sentiment toward the product or purchase:
- positive: the reviewer is mainly satisfied
- negative: the reviewer is mainly dissatisfied
- neutral: the review is mixed, average/mediocre, or expresses no clear
satisfaction or dissatisfaction
Return only a JSON object of the form
{"results": [{"id": "<id>", "label": "positive|negative|neutral"}]}
containing every given id exactly once, in the same order.
Do not add explanations."""

class GunlukLimit(Exception):
    pass

def bekleme_suresi(mesaj):
    """Groq mesajındaki 'Please try again in 1m54.9s' kısmını saniyeye çevirir (bulamazsa None)."""
    m = re.search(r"try again in (?:(\d+)h)?(?:(\d+)m)?(?:([\d.]+)s)?", mesaj)
    if not m or not any(m.groups()):
        return None
    h, dk, sn = m.groups()
    return int(h or 0) * 3600 + int(dk or 0) * 60 + float(sn or 0)

def istek_gonder(grup):
    """Grubu gönderir. Dakikalık 429 → 60 sn bekle. Günlük kota (TPD) → Groq'un söylediği süre kadar bekle (en çok 1 saat).
    Süre okunamazsa ya da 1 saatten uzunsa GunlukLimit hatasıyla durur."""
    user_msg = "Reviews:\n" + "\n".join(f"[{i}] {t}" for i, t in zip(grup["id"], grup["text"]))
    deneme = 0
    while deneme < 10:
        try:
            bas = time.time()
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": user_msg}],
                temperature=0,
                response_format={"type": "json_object"},
                extra_body={"reasoning_effort": REASONING},
            )
            return resp, time.time() - bas
        except groq.RateLimitError as e:
            mesaj = str(e)
            if "per day" in mesaj or "TPD" in mesaj or "RPD" in mesaj:
                sure = bekleme_suresi(mesaj)
                if sure is None or sure > 3600:
                    raise GunlukLimit(mesaj)
                print(f"   {time.strftime('%H:%M')} günlük kota dolu → {sure/60:.1f} dk bekleniyor (Groq'un söylediği süre)")
                time.sleep(sure + 5)
                continue                      # günlük kota beklemeleri deneme hakkından düşmez
            deneme += 1
            print(f"   429 (dakikalık limit), 60 sn bekleniyor... {mesaj[:200]}")
            time.sleep(60)
        except (groq.APIConnectionError, groq.InternalServerError) as e:
            deneme += 1
            print(f"   Bağlantı/sunucu hatası, 30 sn bekleniyor... {str(e)[:200]}")
            time.sleep(30)
    raise RuntimeError("10 denemede de cevap alınamadı; döngü durduruldu. Hücreyi tekrar çalıştırabilirsin.")

def dogrula(metin, beklenen_idler):
    """Geçerliyse ({id: etiket}, None), değilse (None, hata açıklaması) döndürür."""
    try:
        sonuc = json.loads(metin)["results"]
    except Exception as e:
        return None, f"JSON açılamadı: {e}"
    idler = [r.get("id") for r in sonuc]
    if idler != list(beklenen_idler):
        return None, f"id listesi uyuşmuyor (beklenen {len(beklenen_idler)}, gelen {len(idler)})"
    gecersiz = [r.get("label") for r in sonuc if r.get("label") not in ETIKETLER]
    if gecersiz:
        return None, f"geçersiz etiket: {gecersiz[:3]}"
    return {r["id"]: r["label"] for r in sonuc}, None

def gonder_dogrula_kaydet(grup, dil, asama, log_yolu):
    """İsteği gönderir, cevabı ham log'a yazar, doğrulama sonucunu döndürür."""
    resp, sure = istek_gonder(grup)
    metin = resp.choices[0].message.content
    tahmin, hata = dogrula(metin, grup["id"])
    kayit = {"dil": dil, "asama": asama, "ids": grup["id"].tolist(), "cevap": metin,
             "prompt_tokens": resp.usage.prompt_tokens, "completion_tokens": resp.usage.completion_tokens,
             "total_tokens": resp.usage.total_tokens, "sure_sn": round(sure, 2),
             "zaman": time.strftime("%Y-%m-%d %H:%M:%S"), "model": MODEL,
             "reasoning_effort": REASONING, "gecerli": hata is None, "hata": hata}
    with open(log_yolu, "a", encoding="utf-8") as f:
        f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
    # RPM ve TPM'e göre bekle: en az 60/RPM sn; bu isteğin token'ı dakikalık bütçenin ne kadarıysa o kadar
    time.sleep(max(60 / RPM, resp.usage.total_tokens / TPM * 60))
    return tahmin, hata

## 2. İstem denemesi (S1)

In [7]:
# S1: İstem denemesi (10 cümle, reasoning_effort low ve medium). Sonuçlar Bölüm 6.4'e yazıldı; tekrar istek gitmemesi için kapalı.
S1_CALISTIR = False
deneme = [
    ("deneme_en_1", "Absolutely love it, works perfectly and arrived early.", "positive"),
    ("deneme_en_2", "Oh great, it broke after two days. Exactly what I wanted.", "negative"),
    ("deneme_en_3", "The sound quality is great but the battery dies in an hour.", "neutral"),
    ("deneme_en_4", "It's okay. Does the job, nothing special.", "neutral"),
    ("deneme_en_5", "Waste of money, the zipper fell off the first time I used it.", "negative"),
    ("deneme_tr_1", "Harika bir ürün, herkese tavsiye ederim.", "positive"),
    ("deneme_tr_2", "Tam üç günde bozuldu, ne kaliteli ürün ama!", "negative"),
    ("deneme_tr_3", "Kumaşı güzel ama dikişleri biraz özensiz.", "neutral"),
    ("deneme_tr_4", "Fiyatına göre idare eder, ne iyi ne kötü.", "neutral"),
    ("deneme_tr_5", "Kargo çok geç geldi ve ürün kırık çıktı.", "negative"),
]
if S1_CALISTIR:
    grup = pd.DataFrame(deneme, columns=["id", "text", "label"])
    for effort in ["low", "medium"]:
        REASONING = effort
        resp, sure = istek_gonder(grup)
        tahmin, hata = dogrula(resp.choices[0].message.content, grup["id"])
        print(effort, "| hata:", hata, "| çıktı token:", resp.usage.completion_tokens)
    REASONING = "low"
else:
    print("S1 kapalı (S1_CALISTIR = False). Kayıtlı sonuç: iki seviyede de 10/10 aynı etiket; çıktı token low=185, medium=592 → low seçildi.")

S1 kapalı (S1_CALISTIR = False). Kayıtlı sonuç: iki seviyede de 10/10 aynı etiket; çıktı token low=185, medium=592 → low seçildi.


## 3. Pilot çalışma (S2)

In [8]:
# S2: Pilot — doğrulama setinden sınıf başına 20 örnek (random_state=1). Pilot log'u varsa API'ye gidilmez, özet log'dan hesaplanır.
pilot = {}
for dil in ["en", "tr"]:
    val = pd.read_csv(f"data/processed/{dil}_val.csv")
    pilot[dil] = (val.groupby("label", group_keys=False)
                     .sample(n=20, random_state=1)
                     .sample(frac=1, random_state=1)
                     .reset_index(drop=True))

PILOT_LOG = "results/logs/llm_pilot_raw.jsonl"
if not os.path.exists(PILOT_LOG):               # yalnızca log yoksa 6 istek gönderilir
    for dil in ["en", "tr"]:
        for g in range(3):
            grup = pilot[dil].iloc[g*20:(g+1)*20]
            resp, sure = istek_gonder(grup)
            metin = resp.choices[0].message.content
            tahmin, hata = dogrula(metin, grup["id"])
            kayit = {"dil": dil, "grup": g + 1, "ids": grup["id"].tolist(), "cevap": metin,
                     "prompt_tokens": resp.usage.prompt_tokens, "completion_tokens": resp.usage.completion_tokens,
                     "total_tokens": resp.usage.total_tokens, "sure_sn": round(sure, 2),
                     "zaman": time.strftime("%Y-%m-%d %H:%M:%S"), "model": MODEL,
                     "reasoning_effort": REASONING, "gecerli": hata is None, "hata": hata}
            with open(PILOT_LOG, "a", encoding="utf-8") as f:
                f.write(json.dumps(kayit, ensure_ascii=False) + "\n")
            time.sleep(10)
else:
    print("Pilot log'u mevcut → API'ye istek gönderilmedi, özet kayıttan hesaplanıyor.\n")

# Özet (log'dan): geçersiz cevap oranı, ortalama token, pilot doğruluk (yalnızca kontrol amaçlı)
log = [json.loads(s) for s in open(PILOT_LOG, encoding="utf-8")]
for dil in ["en", "tr"]:
    kayitlar = [k for k in log if k["dil"] == dil]
    assert sum(len(k["ids"]) for k in kayitlar) == 60 and set(i for k in kayitlar for i in k["ids"]) == set(pilot[dil]["id"]), \
        f"{dil}: log'daki id'ler bu örneklemle uyuşmuyor"
    gercek = dict(zip(pilot[dil]["id"], pilot[dil]["label"]))
    dogru = sum(r["label"] == gercek[r["id"]] for k in kayitlar if k["gecerli"] for r in json.loads(k["cevap"])["results"])
    gecersiz = sum(not k["gecerli"] for k in kayitlar) / len(kayitlar)
    ort = sum(k["total_tokens"] for k in kayitlar) / len(kayitlar)
    print(f"{dil}: geçersiz cevap oranı = {gecersiz:.1%} | ort. toplam token/istek = {ort:.0f} | pilot doğruluk = {dogru}/60 (yalnızca kontrol amaçlı)")

Pilot log'u mevcut → API'ye istek gönderilmedi, özet kayıttan hesaplanıyor.

en: geçersiz cevap oranı = 0.0% | ort. toplam token/istek = 1561 | pilot doğruluk = 47/60 (yalnızca kontrol amaçlı)
tr: geçersiz cevap oranı = 0.0% | ort. toplam token/istek = 1445 | pilot doğruluk = 45/60 (yalnızca kontrol amaçlı)


## 4. Toplu çalıştırma (S4/S5)

In [9]:
# S4/S5: Test setini 20'şerli gruplar hâlinde sınıflandır. _partial.csv tamamsa API'ye istek gitmez (checkpoint).
def github_ara_kayit(dil):
    """_partial.csv ve ham log dosyasını GitHub'a gönderir; Colab sıfırlanırsa ilerleme kaybolmaz."""
    import subprocess
    komutlar = [["git", "add", f"predictions/llm_{dil}_partial.csv", f"results/logs/llm_{dil}_raw.jsonl"],
                ["git", "commit", "-q", "-m", f"[Sude] LLM {dil} ara kayıt"],
                ["git", "pull", "-q", "--rebase"],
                ["git", "push", "-q"]]
    for k in komutlar:
        r = subprocess.run(k, capture_output=True, text=True)
        if r.returncode != 0 and k[1] != "commit":     # commit 'değişiklik yok' derse sorun değil
            print(f"   ⚠️ GitHub ara kaydı yapılamadı ({' '.join(k[:2])}): {(r.stderr or r.stdout)[:200]}")
            return
    print(f"   {time.strftime('%H:%M')} GitHub ara kaydı yapıldı")


def calistir(dil):
    test = pd.read_csv(f"data/processed/{dil}_test.csv")
    partial = f"predictions/llm_{dil}_partial.csv"
    log_yolu = f"results/logs/llm_{dil}_raw.jsonl"
    os.makedirs("predictions", exist_ok=True); os.makedirs("results/logs", exist_ok=True)

    bitenler = set(pd.read_csv(partial)["id"]) if os.path.exists(partial) else set()
    kalan = test[~test["id"].isin(bitenler)].reset_index(drop=True)
    print(f"[{dil}] tamamlanmış: {len(bitenler)} / {len(test)} | kalan: {len(kalan)}")

    grup_no = 0
    try:
        for bas in range(0, len(kalan), GRUP):
            grup = kalan.iloc[bas:bas + GRUP]
            tahmin, hata = gonder_dogrula_kaydet(grup, dil, "grup", log_yolu)
            if hata:                                   # 1) grubu bir kez tekrar gönder
                print(f"   Geçersiz cevap ({hata}) → grup tekrar gönderiliyor")
                tahmin, hata = gonder_dogrula_kaydet(grup, dil, "grup_tekrar", log_yolu)
            if hata:                                   # 2) yine olmazsa yorumları tek tek gönder
                print(f"   Yine geçersiz ({hata}) → {len(grup)} yorum tek tek gönderiliyor")
                tahmin = {}
                for _, satir in grup.iterrows():
                    tek = grup[grup["id"] == satir["id"]]
                    t, h = gonder_dogrula_kaydet(tek, dil, "tekli", log_yolu)
                    if h:
                        print(f"   ⚠️ {satir['id']} tek başına da geçersiz ({h}); atlandı, sonraki çalıştırmada tekrar denenir")
                    else:
                        tahmin.update(t)
            if tahmin:                                 # sonucu hemen _partial.csv'ye ekle
                pd.DataFrame({"id": list(tahmin.keys()), "pred_label": list(tahmin.values())}).to_csv(
                    partial, mode="a", header=not os.path.exists(partial), index=False)
            grup_no += 1
            if grup_no % 10 == 0:
                github_ara_kayit(dil)
            print(f"   {time.strftime('%H:%M')}  {len(bitenler) + bas + len(grup):,} / {len(test):,} işlendi".replace(",", "."))
    except GunlukLimit as e:
        print(f"\n⛔ GÜNLÜK LİMİT DOLDU. Döngü durduruldu. Groq mesajı:\n{e}\n"
              f"→ Mesajdaki süre dolunca kurulum hücresini ve bu hücreyi tekrar çalıştır; kaldığı yerden devam eder.")

    github_ara_kayit(dil)
    # Durum ve (tamamsa) son dosya
    sonuc = pd.read_csv(partial).drop_duplicates("id", keep="last") if os.path.exists(partial) else pd.DataFrame(columns=["id", "pred_label"])
    print(f"[{dil}] şu an tamamlanan: {len(sonuc)} / {len(test)}")
    if len(sonuc) == len(test):
        son = test.merge(sonuc, on="id", how="left")          # test dosyasının sırası korunur
        son = son.rename(columns={"label": "true_label"})[["id", "text", "true_label", "pred_label"]]
        son.to_csv(f"predictions/llm_{dil}.csv", index=False)
        print(f"✅ predictions/llm_{dil}.csv yazıldı: {son.shape}, boş tahmin: {son['pred_label'].isna().sum()}")

for dil in ["en", "tr"]:      # İngilizce bitince Türkçeye kendiliğinden geçer
    calistir(dil)

[en] tamamlanmış: 3000 / 3000 | kalan: 0
   11:21 GitHub ara kaydı yapıldı
[en] şu an tamamlanan: 3000 / 3000
✅ predictions/llm_en.csv yazıldı: (3000, 4), boş tahmin: 0
[tr] tamamlanmış: 3000 / 3000 | kalan: 0
   11:21 GitHub ara kaydı yapıldı
[tr] şu an tamamlanan: 3000 / 3000
✅ predictions/llm_tr.csv yazıldı: (3000, 4), boş tahmin: 0


## 5. Doğrulama ve çalışma özeti (S6)

In [10]:
# S6: Bir dilin LLM sonuçlarını doğrula, predictions/llm_{dil}.csv dosyasını üret ve çalışma özetine satır ekle
import pandas as pd, json, os

def s6_dogrula_ve_ozetle(dil):
    test = pd.read_csv(f"data/processed/{dil}_test.csv")
    partial = pd.read_csv(f"predictions/llm_{dil}_partial.csv").drop_duplicates("id", keep="last")

    # 1) Son dosyayı test sırasına göre oluştur
    son = test.merge(partial, on="id", how="left").rename(columns={"label": "true_label"})
    son = son[["id", "text", "true_label", "pred_label"]]

    # 2) Kontroller
    kontroller = {
        "satır sayısı 3000":            len(son) == 3000,
        "id'ler test ile aynı sırada":  (son["id"].values == test["id"].values).all(),
        "tekrar eden id yok":           not son["id"].duplicated().any(),
        "boş tahmin yok":               son["pred_label"].notna().all(),
        "etiketler yalnızca 3 sınıf":   son["pred_label"].isin(["negative", "neutral", "positive"]).all(),
        "test dışı id yok":             partial["id"].isin(test["id"]).all(),
    }
    for ad, ok in kontroller.items():
        print(f"   {'✅' if ok else '❌'} {ad}")
    if not all(kontroller.values()):
        print(f"❌ {dil}: kontrol başarısız, dosya YAZILMADI.")
        return
    son.to_csv(f"predictions/llm_{dil}.csv", index=False)
    print(f"✅ predictions/llm_{dil}.csv yazıldı {son.shape}")

    # 3) Çalışma özeti (ham log'dan; pilot hariç)
    log = pd.DataFrame([json.loads(s) for s in open(f"results/logs/llm_{dil}_raw.jsonl", encoding="utf-8")])
    zaman = pd.to_datetime(log["zaman"])
    satir = {
        "dil": dil, "model": log["model"].iloc[0], "reasoning_effort": log["reasoning_effort"].iloc[0],
        "grup_boyutu": 20, "ornek_sayisi": len(son),
        "toplam_istek": len(log),
        "grup_istegi": int((log["asama"] == "grup").sum()),
        "tekrar_gonderilen_grup": int((log["asama"] == "grup_tekrar").sum()),
        "tek_tek_gonderilen_yorum": int((log["asama"] == "tekli").sum()),
        "gecersiz_cevap": int((~log["gecerli"]).sum()),
        "girdi_token": int(log["prompt_tokens"].sum()),
        "cikti_token": int(log["completion_tokens"].sum()),
        "toplam_token": int(log["total_tokens"].sum()),
        "baslangic_utc": str(zaman.min()), "bitis_utc": str(zaman.max()),
        "sure_saat": round((zaman.max() - zaman.min()).total_seconds() / 3600, 2),
        "maliyet_usd": 0.0,   # Groq ücretsiz katman
    }
    yol = "results/logs/llm_calisma_ozeti.csv"
    ozet = pd.read_csv(yol) if os.path.exists(yol) else pd.DataFrame()
    if len(ozet):
        ozet = ozet[ozet["dil"] != dil]          # aynı dil tekrar çalıştırılırsa eski satırı değiştir
    ozet = pd.concat([ozet, pd.DataFrame([satir])], ignore_index=True)
    ozet.to_csv(yol, index=False)
    print("\n--- Çalışma özeti ---")
    print(pd.DataFrame([satir]).T.to_string(header=False))

for dil in ["en", "tr"]:
    s6_dogrula_ve_ozetle(dil)

   ✅ satır sayısı 3000
   ✅ id'ler test ile aynı sırada
   ✅ tekrar eden id yok
   ✅ boş tahmin yok
   ✅ etiketler yalnızca 3 sınıf
   ✅ test dışı id yok
✅ predictions/llm_en.csv yazıldı (3000, 4)

--- Çalışma özeti ---
dil                                        en
model                     openai/gpt-oss-120b
reasoning_effort                          low
grup_boyutu                                20
ornek_sayisi                             3000
toplam_istek                              150
grup_istegi                               150
tekrar_gonderilen_grup                      0
tek_tek_gonderilen_yorum                    0
gecersiz_cevap                              0
girdi_token                            164303
cikti_token                             72764
toplam_token                           237067
baslangic_utc             2026-11-17 15:12:26
bitis_utc                 2026-11-17 20:26:51
sure_saat                                5.24
maliyet_usd                               0.

## 6. Yöntemler arası uyum ve hata örnekleri (S7)

In [11]:
# S7-1: Yöntemler arası uyum — VADER, BERT ve LLM tahminlerini id üzerinden birleştir, doğru/yanlış örtüşmesini say
import pandas as pd

DOSYALAR = {
    "en": {"VADER": "predictions/vader_en.csv",    "BERT": "predictions/bert_en.csv", "LLM": "predictions/llm_en.csv"},
    "tr": {"VADER": "predictions/vader_tr_mt.csv", "BERT": "predictions/bert_tr.csv", "LLM": "predictions/llm_tr.csv"},  # Türkçede VADER çeviri sonrası
}

satirlar, birlesik = [], {}
for dil, yollar in DOSYALAR.items():
    df = None
    for yontem, yol in yollar.items():
        d = pd.read_csv(yol)[["id", "true_label", "pred_label"]].rename(columns={"pred_label": yontem})
        df = d if df is None else df.merge(d, on=["id", "true_label"], how="inner")
    assert len(df) == 3000, f"{dil}: birleştirme sonrası {len(df)} satır (3000 olmalı) — id veya true_label uyuşmuyor"
    v, b, l = (df[y] == df["true_label"] for y in ["VADER", "BERT", "LLM"])
    sayilar = {
        "ucu_de_dogru":          (v & b & l).sum(),
        "ucu_de_yanlis":         (~v & ~b & ~l).sum(),
        "yalnizca_bert_dogru":   (~v & b & ~l).sum(),
        "yalnizca_llm_dogru":    (~v & ~b & l).sum(),
        "yalnizca_vader_dogru":  (v & ~b & ~l).sum(),
        "bert_ve_llm_dogru_vader_yanlis": (~v & b & l).sum(),
    }
    satir = {"dil": dil, "ornek_sayisi": len(df)}
    for ad, n in sayilar.items():
        satir[ad] = int(n)
        satir[ad + "_oran"] = round(n / len(df), 3)
    satirlar.append(satir)
    birlesik[dil] = df.assign(vader_dogru=v, bert_dogru=b, llm_dogru=l)

uyum = pd.DataFrame(satirlar)
uyum.to_csv("results/tables/yontem_uyumu.csv", index=False, encoding="utf-8")
print("✅ results/tables/yontem_uyumu.csv yazıldı\n")
print(uyum.set_index("dil").T.to_string())

# Üçünün de yanlış bildiği örneklerden her dilde 3 tanesine göz at (etiket gürültüsü olabilir)
for dil, df in birlesik.items():
    metin = pd.read_csv(DOSYALAR[dil]["LLM"])[["id", "text"]]
    yanlis = df[~df.vader_dogru & ~df.bert_dogru & ~df.llm_dogru].merge(metin, on="id").sample(3, random_state=1)
    print(f"\n--- {dil}: üçünün de yanlış bildiği örneklerden 3 tanesi ---")
    for _, r in yanlis.iterrows():
        print(f"[{r.id}] gerçek={r.true_label} | VADER={r.VADER} BERT={r.BERT} LLM={r.LLM}\n   {r.text[:300]}")

✅ results/tables/yontem_uyumu.csv yazıldı

dil                                        en        tr
ornek_sayisi                         3000.000  3000.000
ucu_de_dogru                         1262.000  1368.000
ucu_de_dogru_oran                       0.421     0.456
ucu_de_yanlis                         291.000   247.000
ucu_de_yanlis_oran                      0.097     0.082
yalnizca_bert_dogru                   332.000   324.000
yalnizca_bert_dogru_oran                0.111     0.108
yalnizca_llm_dogru                    126.000    99.000
yalnizca_llm_dogru_oran                 0.042     0.033
yalnizca_vader_dogru                  105.000    78.000
yalnizca_vader_dogru_oran               0.035     0.026
bert_ve_llm_dogru_vader_yanlis        725.000   692.000
bert_ve_llm_dogru_vader_yanlis_oran     0.242     0.231

--- en: üçünün de yanlış bildiği örneklerden 3 tanesi ---
[en_045735] gerçek=positive | VADER=negative BERT=neutral LLM=negative
   I bought this fixing powder because it s

In [12]:
# S7-2: LLM hata örnekleri (3 İngilizce + 3 Türkçe) → results/hata_ornekleri/llm.csv (sütunlar bert.csv ile aynı)
import pandas as pd

ORNEKLER = {
    "en": {
        "en_168517": "Gerçek sınıfa göre en sık hata türü (Nötr→Negatif). Hafif bir eleştiri içeren 3 yıldızlı yorum, model tarafından memnuniyetsizlik olarak okunmuştur.",
        "en_134541": "Yorumcu puanını metinde '3.5-4 stars' olarak belirtmiştir; model övgü ifadelerine ağırlık verip Pozitif tahmin etmiştir. Karışık yorumlarda sınırın belirsizliğini gösterir.",
        "en_376587": "Metin yalnızca bir paketleme sorununu anlatmaktadır ve olumlu bir ifade içermemektedir; etiket ise Pozitiftir. Etiket gürültüsü olasılığı yüksektir; BERT de aynı örneği Negatif tahmin etmiştir.",
    },
    "tr": {
        "tr_050272": "İroni: 'çok şaşırttınız beni teşekkürler' ifadesi alaycı biçimde kullanılmıştır. Model teşekkür ifadesini olumlu okuyup Pozitif tahmin etmiş, BERT ise Negatif olarak doğru bilmiştir.",
        "tr_065433": "Gerçek sınıfa göre en sık hata türü (Nötr→Negatif). Yorumda birden fazla şikâyet sıralandığı için model yorumu memnuniyetsizlik olarak sınıflandırmıştır.",
        "tr_065618": "Metin açıkça olumludur ('herkese tavsiye ederim'), ancak etiket Nötrdür. Türkçede sık görülen Nötr→Pozitif hatalarının bir kısmının etiket gürültüsünden kaynaklanabileceğini gösterir; BERT de Pozitif tahmin etmiştir.",
    },
}

parcalar = []
for dil, secim in ORNEKLER.items():
    llm = pd.read_csv(f"predictions/llm_{dil}.csv").set_index("id").loc[list(secim)].reset_index()
    assert (llm["true_label"] != llm["pred_label"]).all(), "Seçilen örneklerden biri yanlış sınıflandırılmış değil!"
    llm["dil"] = dil.upper()
    llm["model"] = "LLM (gpt-oss-120b)"
    llm["aciklama"] = llm["id"].map(secim)
    parcalar.append(llm)

hata = pd.concat(parcalar)[["id", "dil", "model", "text", "true_label", "pred_label", "aciklama"]]
hata.to_csv("results/hata_ornekleri/llm.csv", index=False, encoding="utf-8")
print("✅ results/hata_ornekleri/llm.csv yazıldı", hata.shape)
print(hata[["id", "dil", "true_label", "pred_label", "text"]].to_string(index=False, max_colwidth=70))

✅ results/hata_ornekleri/llm.csv yazıldı (6, 7)
       id dil true_label pred_label                                                                   text
en_168517  EN    neutral   negative Just started using but doesn't seem to last as long as other deodor...
en_134541  EN    neutral   positive Good coverage and great color options. 3.5-4 stars since a couple b...
en_376587  EN   positive   negative One of the four lipglosses was open. The safety plastic was ripped ...
tr_050272  TR   negative   positive Valla bu kadr kötü olacağını tahmin etmiyordum çok şaşırttınız beni...
tr_065433  TR    neutral   negative Küçük olduğunu bile bile aldım onda sorun yok da, çok cıvık geldi b...
tr_065618  TR    neutral   positive                          ürün gayet işimi görül herkese tavsiye ederim


## 7. GitHub'a gönderme

In [13]:
# Oturum sonunda kendi dosyalarımı GitHub'a gönder (yeniden çalıştırmada istemeden push olmasın diye kapalı)
GONDER = False
if GONDER:
    !git add predictions/llm_en.csv predictions/llm_tr.csv predictions/llm_en_partial.csv predictions/llm_tr_partial.csv results/logs/llm_*.jsonl results/logs/llm_calisma_ozeti.csv results/tables/yontem_uyumu.csv results/hata_ornekleri/llm.csv
    !git commit -m "[Sude] LLM dosyaları güncellendi"
    !git pull --rebase
    !git push
else:
    print("GONDER = False → GitHub'a gönderilmedi.")

GONDER = False → GitHub'a gönderilmedi.
